<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.5 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import pandas as pd
import torch
import faiss
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files

print("Loading embedding model...")
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Loading language model...")
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Device:", device)
print("Financial RAG Assistant models loaded successfully!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device: cpu
Financial RAG Assistant models loaded successfully!


In [3]:
from pathlib import Path
from pypdf import PdfReader

def read_document(file_path):
    ext = Path(file_path).suffix.lower()
    text_parts = []

    try:
        if ext == ".pdf":
            reader = PdfReader(file_path)

            for page_number, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                if text.strip():
                    text_parts.append(
                        f"[Page {page_number}]\n{text}"
                    )

        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8",
                      errors="ignore") as f:
                text_parts.append(f.read())

        elif ext == ".csv":
            df = pd.read_csv(file_path)
            text_parts.append(
                "Financial table:\n" +
                df.to_string(index=False)
            )

        elif ext in [".xlsx", ".xls"]:
            sheets = pd.read_excel(file_path, sheet_name=None)

            for sheet_name, df in sheets.items():
                text_parts.append(
                    f"Sheet: {sheet_name}\n" +
                    df.to_string(index=False)
                )

        else:
            return None, f"Unsupported file type: {ext}"

        full_text = "\n\n".join(text_parts).strip()

        if not full_text:
            return None, "No readable text found in this file."

        return full_text, None

    except Exception as e:
        return None, f"Could not read {Path(file_path).name}: {e}"

print("Document reader ready.")

Document reader ready.


In [4]:
knowledge_chunks = []
knowledge_sources = []
faiss_index = None

def build_knowledge_base(file_paths):
    global knowledge_chunks, knowledge_sources, faiss_index

    knowledge_chunks = []
    knowledge_sources = []
    faiss_index = None

    if not file_paths:
        return "Please upload at least one financial document."

    successful_files = []
    errors = []

    for file_path in file_paths:
        text, error = read_document(file_path)
        filename = Path(file_path).name

        if error:
            errors.append(f"{filename}: {error}")
            continue

        chunks = split_into_chunks(text)

        for chunk_number, chunk in enumerate(chunks, start=1):
            knowledge_chunks.append(chunk)
            knowledge_sources.append({
                "file": filename,
                "chunk": chunk_number
            })

        successful_files.append(filename)

    if not knowledge_chunks:
        details = "\n".join(errors)
        return "No readable documents were indexed.\n" + details

    embeddings = embedding_model.encode(
        knowledge_chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embeddings = np.asarray(embeddings, dtype="float32")

    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)

    message = (
        f"Knowledge base ready!\n"
        f"Documents indexed: {len(successful_files)}\n"
        f"Text chunks: {len(knowledge_chunks)}\n"
        f"Chunks are searchable using semantic similarity."
    )

    if successful_files:
        message += "\nFiles: " + ", ".join(successful_files)

    if errors:
        message += "\n\nSkipped files:\n" + "\n".join(errors)

    return message

print("FAISS knowledge-base builder ready.")

FAISS knowledge-base builder ready.


In [5]:
def generate_answer(question, retrieved_docs):
    if not retrieved_docs:
        return (
            "I could not find relevant information in the uploaded "
            "financial documents. Please upload a relevant report "
            "or ask a question covered by the documents."
        )

    context_parts = []

    for i, item in enumerate(retrieved_docs, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, "
            f"chunk {item['chunk']}]\n{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # Keep input within a practical token budget.
    prompt = f"""
You are a financial document analysis assistant.

Answer the question using ONLY the context below.
Do not invent figures, dates, companies, or financial facts.
If the context does not contain enough information, clearly
say that the uploaded documents do not provide enough information.
Explain calculations when relevant. Use simple English.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            num_beams=2,
            do_sample=False,
            early_stopping=True
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    if not answer:
        answer = "I could not generate a reliable answer from the retrieved context."

    return answer

In [6]:
def financial_calculator(revenue, expenses, previous_revenue=None):
    revenue = float(revenue)
    expenses = float(expenses)

    if revenue < 0 or expenses < 0:
        raise ValueError("Revenue and expenses cannot be negative.")

    profit = revenue - expenses
    profit_margin = (profit / revenue * 100) if revenue else None

    result = {
        "Revenue": round(revenue, 2),
        "Expenses": round(expenses, 2),
        "Profit / Loss": round(profit, 2),
        "Profit Margin (%)": (
            round(profit_margin, 2)
            if profit_margin is not None else "N/A"
        )
    }

    if previous_revenue is not None:
        previous_revenue = float(previous_revenue)

        if previous_revenue < 0:
            raise ValueError("Previous revenue cannot be negative.")

        result["Revenue Growth (%)"] = (
            round(
                (revenue - previous_revenue) /
                previous_revenue * 100, 2
            )
            if previous_revenue else "N/A"
        )

    return result


# Test example
print(financial_calculator(
    revenue=500000,
    expenses=350000,
    previous_revenue=400000
))

{'Revenue': 500000.0, 'Expenses': 350000.0, 'Profit / Loss': 150000.0, 'Profit Margin (%)': 30.0, 'Revenue Growth (%)': 25.0}


In [ ]:
def answer_question(question):
    if not question or not question.strip():
        return "Please enter a question.", ""

    if faiss_index is None:
        return "Please upload documents and click Build Knowledge Base first.", ""

    retrieved = retrieve_documents(question, top_k=4)
    answer = generate_answer(question, retrieved)

    source_lines = []

    for i, item in enumerate(retrieved, start=1):
        source_lines.append(
            f"{i}. {item['source']} | "
            f"Chunk {item['chunk']} | "
            f"Similarity: {item['score']:.3f}\n"
            f"Excerpt: {item['text'][:500]}"
        )

    sources = "\n\n".join(source_lines)
    return answer, sources


with gr.Blocks(title="Financial RAG Assistant") as demo:
    gr.Markdown("# Financial RAG Assistant")
    gr.Markdown(
        "Upload financial reports and ask questions about "
        "revenue, profit, expenses, risks, and financial performance."
    )

    document_upload = gr.File(
        label="Upload financial documents",
        file_count="multiple",
        file_types=[".pdf", ".txt", ".csv", ".xlsx", ".xls"],
        type="filepath"
    )

    build_button = gr.Button("Build Knowledge Base")
    status_output = gr.Textbox(
        label="Knowledge Base Status",
        lines=4
    )

    build_button.click(
        fn=build_knowledge_base,
        inputs=[document_upload],
        outputs=[status_output]
    )

    question_input = gr.Textbox(
        label="Ask a financial question",
        placeholder="Example: What was the company's revenue?",
        lines=2
    )

    ask_button = gr.Button("Ask Question")

    answer_output = gr.Textbox(
        label="AI Answer",
        lines=6
    )

    source_output = gr.Textbox(
        label="Retrieved Sources and Evidence",
        lines=12
    )

    ask_button.click(
        fn=answer_question,
        inputs=[question_input],
        outputs=[answer_output, source_output]
    )

    gr.Examples(
        examples=[
            ["What was the company's total revenue?"],
            ["What are the main expenses?"],
            ["What financial risks are mentioned in the report?"],
            ["Summarize the company's financial performance."]
        ],
        inputs=[question_input]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d69a479ab54520bb12.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
sample_report = """
SAMPLE COMPANY FINANCIAL REPORT - FY 2025

Note: All figures are fictional and created for demonstration.

Total revenue for FY 2025 was INR 500000.
Total operating expenses for FY 2025 were INR 350000.
The previous year's revenue was INR 400000.

The company's main expenses include salaries, software,
marketing, rent, and operational costs.

The company earned a profit before tax of INR 150000,
based on revenue minus the expenses provided in this sample.

Key risks include rising operating costs, competition,
customer demand fluctuations, and delayed customer payments.

Management plans to improve cost control and increase
customer retention in the next financial year.
"""

with open("sample_financial_report.txt", "w", encoding="utf-8") as f:
    f.write(sample_report)

print("Sample report created: sample_financial_report.txt")